In [1]:
import glob
import time
import cv2
import torch
import json
from pathlib import Path
from mmpose.apis import init_model, inference_topdown
from mmpose.visualization import PoseLocalVisualizer
from mmdet.apis import init_detector, inference_detector
from enum import Enum
from typing import List

d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmengine\optim\optimizer\zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [3]:
CONFIG_ROOT_PATH = "D:\\Magistrska\\mmpose\\configs\\"
CHECKPOINT_ROOT_PATH = "D:\\Magistrska\\mmpose\\checkpoints\\"
EXPORT_PATH_ROOT = "D:\\Magistrska\\code\\exported_models\\"

KEYPOINT_IDS = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

class Keypoint:
    def __init__(self, id, x, y):
        self.id = id
        self.pixelPosition = (x, y)

class PoseResult:
    def __init__(self, file, keypoints):
        self.file = file
        self.keypoints = keypoints


class PoseModel(Enum):
    RTMO_L = 1
    RTMO_M = 2
    RTMO_S = 3
    RTMO_T = 4
    RTMO_X = 5


class DetectorModel(Enum):
    YOLOX_L = 1
    YOLOX_M = 2
    YOLOX_S = 3
    YOLOX_T = 4


def get_cfg_and_checkpoint(option):
    if option == PoseModel.RTMO_L:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\body7\\rtmo-l_16xb16-600e_body7-640x640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\body7\\rtmo-l_16xb16-600e_body7-640x640-b37118ce_20231211.pth"
    elif option == PoseModel.RTMO_M:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\body7\\rtmo-m_16xb16-600e_body7-640x640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\body7\\rtmo-m_16xb16-600e_body7-640x640-39e78cc4_20231211.pth"
    elif option == PoseModel.RTMO_S:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\body7\\rtmo-s_8xb32-600e_body7-640x640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\body7\\rtmo-s_8xb32-600e_body7-640x640-dac2bf74_20231211.pth"
    elif option == PoseModel.RTMO_T:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\body7\\rtmo-t_8xb32-600e_body7-416x416.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\body7\\rtmo-t_8xb32-600e_body7-416x416-f48f75cb_20231219.pth"
    elif option == PoseModel.RTMO_X:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\coco\\rtmo-x_8xb256-700e_coco-384x288.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\coco\\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth"

    elif option == DetectorModel.YOLOX_L:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_l_8xb32-300e_coco-640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth"
    elif option == DetectorModel.YOLOX_M:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_m_8xb32-300e_coco-640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_m_8xb32-300e_coco-640-84e9a538_20230829.pth"
    elif option == DetectorModel.YOLOX_S:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_s_8xb32-300e_coco-640.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_s_8xb32-300e_coco-640-56c79c1f_20230829.pth"
    elif option == DetectorModel.YOLOX_T:
        cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_tiny_4xb64-300e_coco-416.py"
        checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_tiny_4xb64-300e_coco-416-76eb44ca_20230829.pth"

    return cfg_path, checkpoint_path

In [4]:
def processKeypoints(keypoints: list, width: int, height: int) -> List[Keypoint]:
    final_keypoints: list[Keypoint] = []
    for i, keypoint in enumerate(keypoints):
        keypoint_id = KEYPOINT_IDS[i]
        x = float(keypoint[0]) / width
        y = float(keypoint[1]) / height
        final_keypoints.append(Keypoint(keypoint_id, x, y))

    return final_keypoints


def savePoseResultsToJson(results: List[PoseResult], output_path: str, model: str):
    keypoints_dict = {
        "model": model,
        "keypointsOnImages": [
            {
                "file": result.file,
                "keypoints": [
                    {
                        "id": keypoint.id,
                        "pixelPosition": {
                            "x": keypoint.pixelPosition[0],
                            "y": keypoint.pixelPosition[1],
                        },
                    }
                    for keypoint in result.keypoints
                ],
            }
            for result in results
        ],
    }

    with open(output_path, "w") as f:
        json.dump(keypoints_dict, f, indent=4)

### Image Recognition

In [6]:
SELECTED_MODEL = PoseModel.RTMO_X
SELECTED_DETECTOR = DetectorModel.YOLOX_L
USE_DETECTOR = True

TITLE='cmj-bb_frames_10fps'
INPUT_DIR = f"D:\\Magistrska\\blindoff-magistrska\\fitcode-frontend-next\\public\\exercise-cut-videos-to-images\\{TITLE}\\images"
OUTPUT_DIR = f"D:\\Magistrska\\blindoff-magistrska\\fitcode-frontend-next\\public\\exercise-cut-videos-to-images\\{TITLE}\\results"
SAVE_VIS = False # True to save visualizations (with keypoint drawings)
SHOW_VIS = False # True to show visualizations
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.tif", "*.tiff")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

# --------- Init pose model ---------
cfg_path, checkpoint_path = get_cfg_and_checkpoint(SELECTED_MODEL)
pose_model = init_model(cfg_path, checkpoint_path, device=device)

# --------- Init detector model ---------
if USE_DETECTOR:
    detector_cfg, detector_ckpt = get_cfg_and_checkpoint(SELECTED_DETECTOR)
    det_model = init_detector(detector_cfg, detector_ckpt, device=device)

# --------- Init visualizer ---------
visualizer = PoseLocalVisualizer()
visualizer.set_dataset_meta(pose_model.dataset_meta)

# ----------------- Collect images -----------------
input_dir = Path(INPUT_DIR)
assert input_dir.exists(), f"INPUT_DIR does not exist: {input_dir}"

image_paths = []
for ext in IMG_EXTS:
    image_paths.extend(glob.glob(str(input_dir / ext)))
image_paths = sorted(image_paths)

if not image_paths:
    raise RuntimeError(f"No images found in {input_dir} with extensions: {IMG_EXTS}")

# ----------------- Output dir -----------------
out_dir = Path(OUTPUT_DIR)
if SAVE_VIS:
    out_dir.mkdir(parents=True, exist_ok=True)

# ----------------- Process loop -----------------
total = len(image_paths)
start_time = time.time()
max_fps = 0.0

pose_results = []

for idx, img_path in enumerate(image_paths, start=1):
    print(f"Processing image ({idx}/{total})")
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"[WARN] Could not read image: {img_path}")
        continue

    # --------- BBoxes ---------
    if USE_DETECTOR:
        det_result = inference_detector(det_model, frame)
        pred_instance = det_result.pred_instances
        person_bboxes = pred_instance.bboxes[pred_instance.labels == 0]
        scores = pred_instance.scores[pred_instance.labels == 0]

        if len(person_bboxes) > 0:
            best_idx = scores.argmax().item()
            bboxes = person_bboxes[best_idx].reshape(1, -1)
        else:
            bboxes = []
    else:
        # Static single-person bbox
        h, w = frame.shape[:2]
        bbox_w = int(w * 0.6)
        bbox_h = int(h * 0.9)
        bbox_x = int((w - bbox_w) / 2)
        bbox_y = int((h - bbox_h) / 2)
        # NOTE: if your inference expects XYXY, change to:
        # bboxes = [[bbox_x, bbox_y, bbox_x + bbox_w, bbox_y + bbox_h]]
        bboxes = [[bbox_x, bbox_y, bbox_w, bbox_h]]

    # --------- Pose inference ---------
    model_pose_results = inference_topdown(pose_model, frame, bboxes)

    # --------- Visualization ---------
    vis_frame = frame.copy()
    pose = model_pose_results[0]
    # for pose in model_pose_results:
    visualizer.add_datasample(
        "result",
        vis_frame,
        data_sample=pose,
        draw_gt=False,
        draw_pred=True,
        show=False,
        wait_time=0,
        out_file=None,
    )
    vis_frame = visualizer.get_image()

    w, h = frame.shape[1], frame.shape[0]
    keypoints = processKeypoints(
        pose.pred_instances.keypoints[0], w, h 
    )
    pose_results.append(PoseResult(file=Path(img_path).name, keypoints=keypoints))


    # --------- Save ---------
    if SAVE_VIS:
        out_path = out_dir / f"{Path(img_path).stem}_pose{Path(img_path).suffix}"
        cv2.imwrite(str(out_path), vis_frame)

    # --------- Show (optional) ---------
    if SHOW_VIS:
        cv2.imshow("RTMO Folder Inference", vis_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("Stopping early.")
            break



# ----------------- Done -----------------
print(f"Processed: {idx}/{total} images")

savePoseResultsToJson(
    pose_results,
    output_path=str(out_dir / "RTMO_results.json"),
    model="RTMO_X"
)

if SHOW_VIS:
    cv2.destroyAllWindows()

cpu
Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\rtmo\coco\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth


d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmengine\runner\checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, 

Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\yolox\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth


d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmdet\apis\inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(


Processing image (1/107)


d:\magistrska\mmpose\mmpose\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWa

Processing image (2/107)
Processing image (3/107)
Processing image (4/107)
Processing image (5/107)
Processing image (6/107)
Processing image (7/107)
Processing image (8/107)
Processing image (9/107)
Processing image (10/107)
Processing image (11/107)
Processing image (12/107)
Processing image (13/107)
Processing image (14/107)
Processing image (15/107)
Processing image (16/107)
Processing image (17/107)
Processing image (18/107)
Processing image (19/107)
Processing image (20/107)
Processing image (21/107)
Processing image (22/107)
Processing image (23/107)
Processing image (24/107)
Processing image (25/107)
Processing image (26/107)
Processing image (27/107)
Processing image (28/107)
Processing image (29/107)
Processing image (30/107)
Processing image (31/107)
Processing image (32/107)
Processing image (33/107)
Processing image (34/107)
Processing image (35/107)
Processing image (36/107)
Processing image (37/107)
Processing image (38/107)
Processing image (39/107)
Processing image (40

### Webcam

In [ ]:
SELECTED_MODEL = PoseModel.RTMO_L
SELECTED_DETECTOR = DetectorModel.YOLOX_S
USE_DETECTOR = True

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

# --------- Init pose model ---------
cfg_path, checkpoint_path = get_cfg_and_checkpoint(SELECTED_MODEL)
pose_model = init_model(cfg_path, checkpoint_path, device=device)

# --------- Init detector model ---------
if USE_DETECTOR:
    detector_cfg, detector_ckpt = get_cfg_and_checkpoint(SELECTED_DETECTOR)
    det_model = init_detector(detector_cfg, detector_ckpt, device=device)

# --------- Init visualizer ---------
visualizer = PoseLocalVisualizer()
visualizer.set_dataset_meta(pose_model.dataset_meta)

# --------- Open webcam ---------
cap = cv2.VideoCapture(0)

frame_count = 0
start_time = time.time()
avg_fps = 0
max_fps = 0
multi_pose_counter = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if USE_DETECTOR:
        # --------- Person detection ---------
        det_result = inference_detector(det_model, frame)
        pred_instance = det_result.pred_instances
        person_bboxes = pred_instance.bboxes[pred_instance.labels == 0]
        scores = pred_instance.scores[pred_instance.labels == 0]

        if len(person_bboxes) > 0:
            best_idx = scores.argmax().item() # Choose the best bbox (e.g., highest confidence)
            bboxes = person_bboxes[best_idx].reshape(1, -1)
        else:
            bboxes = []
    else:
        # --------- Static single-person bbox (skip detector) ---------
        h, w = frame.shape[:2]
        bbox_w = int(w * 0.6)
        bbox_h = int(h * 0.9)
        bbox_x = int((w - bbox_w) / 2)
        bbox_y = int((h - bbox_h) / 2)

        bboxes = [[bbox_x, bbox_y, bbox_w, bbox_h]]

    # --------- Pose inference ---------
    model_pose_results = inference_topdown(pose_model, frame, bboxes)

    # --------- Visualization ---------
    vis_frame = frame.copy()
    for pose in model_pose_results:
        visualizer.add_datasample(
            'result',
            vis_frame,
            data_sample=pose,
            draw_gt=False,
            draw_pred=True,
            show=False,
            wait_time=1,
            out_file=None
        )
        vis_frame = visualizer.get_image()

    # --------- FPS display ---------
    frame_count += 1
    elapsed = time.time() - start_time
    fps = frame_count / elapsed
    if fps > max_fps:
        max_fps = fps
    avg_fps = (avg_fps * (frame_count - 1) + fps) / frame_count
    cv2.putText(vis_frame, f'FPS: {fps:.2f}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(vis_frame, f'Avg FPS: {avg_fps:.2f}', (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(vis_frame, f'Max FPS: {max_fps:.2f}', (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(vis_frame, f'Selected Model: {SELECTED_MODEL}', (10, 120),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # --------- Show frame ---------
    # fullscren
    cv2.imshow('RTMO Webcam Test', vis_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('Model:', SELECTED_MODEL)
        print('Detector:', SELECTED_DETECTOR)
        print(f'Average FPS: {avg_fps:.2f}')
        print(f'Max FPS: {max_fps:.2f}')
        break

cap.release()
cv2.destroyAllWindows()

cuda


d:\magistrska\mmpose\mmpose\datasets\datasets\utils.py:102: UserWarning: The metainfo config file "configs/_base_/datasets/coco.py" does not exist. A matched config file "d:\magistrska\mmpose\mmpose\.mim\configs\_base_\datasets\coco.py" will be used instead.
  warnings.warn(
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmengine\runner\checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch

Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\rtmo\body7\rtmo-l_16xb16-600e_body7-640x640-b37118ce_20231211.pth
Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\yolox\yoloxpose_s_8xb32-300e_coco-640-56c79c1f_20230829.pth


d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\mmdet\apis\inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
d:\magistrska\mmpose\mmpose\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
d:\ProgramFiles\Anaconda\envs\openmmlab\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Model: PoseModel.RTMO_L
Detector: DetectorModel.YOLOX_S
Average FPS: 5.35
Max FPS: 5.79


## 🧪 RTMO Webcam Inference FPS Single-pose vs Multi-pose Results on GPU

#### Detector: YOLOX-s

Webcam FPS = 30

SP = single-pose, MP = multi-pose

| Model     | Config File                                       | Checkpoint File                            | Input Size | FPS (Avg) SP | FPS (Max) SP | FPS (Avg) MP | FPS (Max) MP |
|-----------|---------------------------------------------------|---------------------------------------------|------------|--------------|--------------|--------------|--------------|
| RTMO-L    | rtmo-l_16xb16-600e_body7-640x640.py               | rtmo-l.pth                                  | 640x640    |   7.26      |   8.04      |      4.65        |     5.25         |
| RTMO-M    | rtmo-m_16xb16-600e_body7-640x640.py               | rtmo-m.pth                                  | 640x640    |     8.28         |      8.66        |     5.65         |     6.55         |
| RTMO-S    | rtmo-s_8xb32-600e_body7-640x640.py                | rtmo-s.pth                                  | 640x640    |     7.69         |     8.40         |      5.61        |     6.25         |
| RTMO-T    | rtmo-t_8xb32-600e_body7-416x416.py                | rtmo-t.pth                                  | 416x416    |      7.85        |      8.51        |      6.02        |      6.51        |